## Housekeeping (Step 1)

In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace <username> and <repo> with the actual values provided by your instructor

# FIX ME: Replace with your corrected BASE_URL
BASE_URL = "https://raw.githubusercontent.com/shiqichen27/CIS3120/main/" # e.g., "https://raw.githubusercontent.com/your_username/your_repo/main/"

BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"


 # Local paths in the Colab /content directory

BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"

## Download CSV Files (Step 2)

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
  urllib.request.urlretrieve(url, path)
  print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


## Create Database and Tables (Step 3)

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

# CREATE TABLE statements go here
#Book Table
conn.execute('''
  CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
  );
''')
#Member Table
conn.execute('''
  CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
  );
''')

#Loan Table
conn.execute('''
  CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
  );
''')

conn.commit()
print("Tables created.")

Tables created.


In [4]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',), ('patrons',), ('sqlite_sequence',)]

## Load Book Data (Step 4)

In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
  reader = csv.DictReader(f)
  for row in reader:
    conn.execute(
        'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
         (row['callNo'], row['title'], row['author'])
        )

conn.commit()

print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


## Load Member Data (Step 5)

In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
  reader = csv.DictReader(f)
  for row in reader:
    conn.execute(
        'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
         (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()

print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


## Load Loan Data (Step 6)

In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
  reader = csv.DictReader(f)
  for row in reader:
    # Convert empty dateReturned to None (→ NULL in SQLite)
    date_returned = row['dateReturned'] if row['dateReturned'].strip() else None
    conn.execute(
        '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
        VALUES (?, ?, ?, ?, ?);''',
         (row['callNo'], int(row['id']),
          row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


## Query 1 - All Books



In [8]:
#Retrieve all columns from the Book table, ordered alphabetically
#by author last name (hint: SQLite has no last-name function — order by the author column as stored).

book = conn.execute('''
    SELECT title, author
    FROM   book
    ORDER BY author;
''')

print("All Books By Author's Name: ")
for row in book:
    print(row)


All Books By Author's Name: 
('Medicine in medieval England.', 'Charles H Talbot')
('Data model patterns : conventions of thought', 'David Hay')
('Atlas of medieval Europe', 'Donald Matthew')
('Medieval women', 'Eileen Power')
('Medieval miscellany', 'Frederick Whitehead')
("Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
("Joe Celko's SQL puzzles & answers", 'Joe Celko')
("Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('Medieval medicine and the plague', 'Lynne Elliott')
('Information modeling and relational databases', 'T A Halpin')
('Programming Android', 'Zigurd R Mednieks')


## Query 2 - Books Not Yet Returned

In [9]:
#Retrieve the title of each book, and the first and last name of the member who borrowed it,
#for all loans where dateReturned is NULL (i.e., the book is still out).

#This query requires a JOIN across all three tables.

book = conn.execute('''
    SELECT book.title, member.firstname, member.lastName
    FROM   book
    JOIN   loan ON book.callNo = loan.callNo
    JOIN   member ON loan.id = member.id
    WHERE  loan.dateReturned IS NULL;

''')

print("Books Not Yet Returned: ")
for row in book:
    print(row)

Books Not Yet Returned: 
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3 — Loan History for a Specific Book

In [10]:
#Retrieve the full loan history for the book with callNo R 141 E45 2006
#— show the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.

book = conn.execute('''
    SELECT member.firstname, member.lastName, loan.dateBorrowed, loan.dateDue, loan.dateReturned
    FROM book
    JOIN loan ON book.callNo = loan.callNo
    JOIN member ON loan.id = member.id
    WHERE book.callNo = 'R 141 E45 2006'
    ORDER BY loan.dateBorrowed ASC;

''')

print("Loan History for a Specific book: ")
for row in book:
    print(row)

Loan History for a Specific book: 
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


##Query 4 - Members Who Have Never Borrowed a Book

In [11]:
#Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.
#Use a LEFT JOIN or a sub-query with NOT IN.

book = conn.execute('''
    SELECT m.id, m.firstname, m.lastName
    FROM member AS m
    LEFT JOIN loan ON m.id = loan.id
    WHERE m.id NOT IN (
      SELECT id
      FROM loan
      WHERE dateBorrowed IS NOT NULL
      )
''')

print("Members Who Have Never Borrowed a Book: ")
for row in book:
    print(row)

Members Who Have Never Borrowed a Book: 
(4, 'John', 'Martin')


## Query 5 - Count of Loans per Member

In [12]:
#Retrieve each member's full name and the total number of loans they have made (including completed ones).
#Include members with zero loans. Order by number of loans descending.

#Hint: Use a LEFT JOIN from Member to Loan combined with COUNT() and GROUP BY.

book = conn.execute('''
    SELECT member.firstname, member.lastName, COUNT(loan.id) AS loan_count
    FROM member
    LEFT JOIN loan ON member.id = loan.id
    GROUP BY member.id
    ORDER BY loan_count DESC;
''')

print("Count of Loans per Member: ")
for row in book:
    print(row)

Count of Loans per Member: 
('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


## Query 6 - The Number of Books Returned By A Member

In [13]:
#Write one original query of your own design that reveals something interesting or useful about this library dataset.
#It must use at least one JOIN and at least one aggregate function or WHERE condition not used in Queries 1–5.
#Precede the query with a Markdown cell that states the business question you are answering and why it is useful.

book = conn.execute('''
    SELECT member.firstname, member.lastName, COUNT(loan.id) AS book_count
    FROM member
    LEFT JOIN loan ON member.id = loan.id
    WHERE loan.dateReturned
''')

print("The Number of Books Returned By A Member: ")
for row in book:
    print(row)

The Number of Books Returned By A Member: 
('John', 'Smith', 2)


#Summary
#### One data quality observation I made is that the SQL statements are a good way to identify who has borrowed and returned books in the library system. It helps keep track of who needs to be reminded to return their books and also see what kinds of books has been loaned. One limitation of this dataset for a real library system is that there are hundreds of records and would require more than one subquery.